# Day 29: Mini Project - Orders DQ Script

**Plan:**
- Data: small orders dataset, with a planted null, duplicate, and out-of-range value
- Checks: null check (order_id), duplicate check (order_id), range check (amount) using config
- Log every check result
- Save one row per check as a Delta report table

In [0]:
config_content = """valid_statuses:
  - Pending
  - Shipped
  - Completed
  - Cancelled

amount_range:
  min: 0
  max: 10000

thresholds:
  null_warning_pct: 5
  null_fail_pct: 20
"""

volume_path = "/Volumes/workspace/default/day29_files"
with open(f"{volume_path}/config.yaml", "w") as f:
    f.write(config_content)

In [0]:
csv_content = """order_id,customer_name,product,quantity,amount,order_date,status
1001,Alice Johnson,Laptop,1,1200.50,2026-08-20,Completed
1002,Bob Smith,Mouse,2,45.00,2026-08-21,Completed
1003,Carol Davis,Keyboard,1,85.00,2026-08-21,Shipped
,David Lee,Monitor,1,350.00,2026-08-22,Pending
1005,Emma Wilson,Desk,1,-75.00,2026-08-23,Pending
1006,Frank Brown,Chair,2,240.00,2026-08-23,Completed
1006,Frank Brown,Chair,2,240.00,2026-08-23,Completed
1007,Grace Miller,Headphones,1,120.00,2026-08-24,Shipped
1008,Henry Taylor,Webcam,1,95.00,2026-08-24,Completed
1009,Ivy Martinez,USB Hub,3,60.00,2026-08-25,Pending
"""

volume_path = "/Volumes/workspace/default/day29_files"
with open(f"{volume_path}/orders.csv", "w") as f:
    f.write(csv_content)

orders = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/orders.csv")
display(orders)

In [0]:
import logging, yaml
from pyspark.sql.functions import col

volume_path = "/Volumes/workspace/default/day29_files"

with open(f"{volume_path}/config.yaml") as f:
    config = yaml.safe_load(f)

logging.basicConfig(
    filename=f"{volume_path}/mini_project.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("mini_project")

In [0]:
from pyspark.sql.functions import count, when
import logging, yaml
from pyspark.sql.functions import col

# Load config
volume_path = "/Volumes/workspace/default/day29_files"
with open(f"{volume_path}/config.yaml") as f:
    config = yaml.safe_load(f)

# Initialize results list
results = []

# Check 1: Null check on order_id
null_count = orders.filter(col("order_id").isNull()).count()
print(f"✓ Null check: found {null_count} nulls in order_id")
results.append(("null_check", "order_id", null_count, "FAIL" if null_count > 0 else "PASS"))

# Check 2: Duplicate check on order_id
duplicate_count = orders.groupBy("order_id").count().filter(col("count") > 1).count()
print(f"✓ Duplicate check: found {duplicate_count} duplicate order_ids")
results.append(("duplicate_check", "order_id", duplicate_count, "FAIL" if duplicate_count > 0 else "PASS"))

# Check 3: Range check on amount (from config)
min_amount = config['amount_range']['min']
max_amount = config['amount_range']['max']
out_of_range = orders.filter((col("amount") < min_amount) | (col("amount") > max_amount)).count()
print(f"✓ Range check: found {out_of_range} amounts outside [{min_amount}, {max_amount}]")
results.append(("range_check", "amount", out_of_range, "FAIL" if out_of_range > 0 else "PASS"))

print(f"\n✓ Completed {len(results)} data quality checks")

In [0]:
# Create results DataFrame
report_df = spark.createDataFrame(results, ["check_type", "column_name", "issue_count", "status"])

# Save to Delta table
report_path = f"{volume_path}/dq_report"
report_df.write.format("delta").mode("overwrite").save(report_path)

print(f"\n📊 Data Quality Report saved to: {report_path}")
print("\nReport Summary:")
display(report_df)